In [56]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [57]:
# T1
task1 = train['Publisher'].nunique()
print(task1)
# T2
goodHeroes = train[train['Alignment'] == 'good']
task2 = goodHeroes.groupby('Publisher')['name'].nunique().idxmax()
print(task2)

23
Marvel Comics


In [58]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV, KFold
from xgboost import XGBRegressor
from scipy.stats import randint, uniform
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline

ohcols = ['Gender', 'Alignment', 'stability_index']
targetcols = ['Publisher', 'Race', 'Hair color', 'Skin color', 'Eye color']
# Bools -> int
boolCols = train.select_dtypes(include='bool').columns
for b in boolCols:
    train[b] = train[b].astype(int)
    test[b] = test[b].astype(int)

X_train1 = train.drop(columns=['powerstats__combat', 'id', 'name', 'Super Strength'])
y_train1 = train['powerstats__combat']
X_test1 = test.drop(columns=['id', 'name'])

preprocessor = ColumnTransformer(
    transformers=[
        ('oneHot', OneHotEncoder(drop='first', handle_unknown='ignore'), ohcols),
        ('target', TargetEncoder(random_state=42, smooth='auto'), targetcols)
    ],
    remainder='passthrough'
)

pipe1 = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBRegressor(random_state=42))
])

param_dist = {
    "model__n_estimators": randint(100, 500),
    "model__max_depth": randint(3, 7),
    "model__learning_rate": uniform(0.01, 0.2),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rs = RandomizedSearchCV(
    estimator=pipe1,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    random_state=42,
    n_jobs=-1,
    scoring='neg_mean_absolute_error'
)

rs.fit(X_train1, y_train1)
model = rs.best_estimator_
predictions1 = model.predict(X_test1)

In [71]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
X_train2 = train.drop(columns=['Super Strength', 'id', 'name'])
y_train2 = train['Super Strength']
test['powerstats__combat'] = predictions1 # use previous predictions to fill column
X_test2 = test.drop(columns=['id', 'name'])

fraud = train[train['Super Strength'] == 1]
notFraud = train[train['Super Strength'] == 0]
scale_pos_weight = notFraud.shape[0] / fraud.shape[0]

pipe2 = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight))
])

cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rs2 = RandomizedSearchCV(
    estimator=pipe2,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv2,
    random_state=42,
    n_jobs=-1,
    scoring='accuracy'
)

rs2.fit(X_train2, y_train2)
model2 = rs2.best_estimator_
predictions2 = model2.predict(X_test2)

In [72]:
rows = []
rows.append({
    'id': "GLOBAL",
    'subtaskID': "task1",
    'answer': int(task1)
})
rows.append({
    'id': "GLOBAL",
    'subtaskID': "task2",
    'answer': task2
})
for id, pred in zip(test['id'], predictions1):
    rows.append({
        'id': id,
        'subtaskID': "task3",
        'answer': round(float(pred), 4)
    })
for id, pred in zip(test['id'], predictions2):
    rows.append({
        'id': id,
        'subtaskID': "task4",
        'answer': pred.astype(int)
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)